# Calculate the baseline mortality rate (BMR)

Processing country-level mortality rate csv files from [GBD Results VizHub](https://vizhub.healthdata.org/gbd-results/) to xarray format. See the README.md file for more details regarding the search terms used to create this data.

BMR is calculated as the average mortality rate over a set period. Here we define this as 1990-2009, the data we extracted only includes these years.
This is calculated for each health outcome.

In [1]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Health outcomes ===
health_vars_out = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
                   "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
                   "STROKE", "DEMENTIA"]

# Abbreviations below are from the renamed VizHub download file names
# Change this if you haven't renamed them with the variable names above
# Add variables in the same order as the health_vars_out to match
# See README.md for more information
health_vars_in = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
                  "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
                  "STROKE", "DEMENTIA"]

In [3]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
GBD_version = "GBD23"

# Loop though each health outcome
for i in range(len(health_vars_in)):
    print(f"Processing health outcome {health_vars_out[i]}")
    health_VAR = health_vars_in[i]
    health_VAR_out = health_vars_out[i]

    # Reads the CSV file into a DataFrame
    # Mortality rate by country
    in_file = os.path.join(BMR_DIR, f"IHME-GBD_20{GBD_version[-2:]}_DATA-{health_VAR}.csv")
    df = pd.read_csv(in_file)

    # Options are Number, Percent or Rate (Rate is per 100,000)
    # We will need to divide by 100,000 later to get rate per 1
    df = df[df["metric_name"] == "Rate"]

    # Calculate the mean, we use the 2015-2019 mean as our baseline period
    df_mean = df.groupby("location_name").mean("year").reset_index()

    country = df_mean["location_name"]
    val = df_mean["val"]  # the mean value [GBD Results Tool User Guide]
    upper = df_mean["upper"]  # 95% Confidence Interval Upper Bound
    lower = df_mean["lower"]  # 95% Confidence Interval Lower Bound

    data = np.stack([lower, val, upper], axis=1)  # shape (204 countries, 3 stats)

    # Create the xarray DataArray
    da = xr.DataArray(
        data,
        dims=["country", "quantile"],
        coords={
            "country": country,
            "quantile": ["lower", "mean", "upper"]
        },
        name="BMR_by_country"
    )

    # Dividing to find rate per 1 person
    # Mortality rate from VizHub download is provided as a RATE per 100,000
    # i.e if the rate was 10 per 100,000 the value provided would be 10, not 0.0001
    # To calculate the mortality we must divide the rate by 100,000 to convert
    # to a per-person basis.
    da = da / 100000

    cite = ("Global Burden of Disease Collaborative Network."
            "Global Burden of Disease Study 2023 (GBD 2023) Results."
            "Seattle, United States: Institute for Health Metrics and " 
            "Evaluation (IHME), 2024. Available from "
            "https://vizhub.healthdata.org/gbd-results/. (Accessed "
            "[17 August 2026])")

    da.attrs["description"] = ("Average Baseline Mortality Rate per person per"
                               f" country for {health_VAR_out} from 2015-2019")
    da.attrs["GBD version"] = GBD_version
    da.attrs["citation"] = cite

    out_file = f"{GBD_version}_BMR_Country_{health_VAR_out}_2015-2019.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    da.to_netcdf(out_path)

print("All processing complete.")

Processing health outcome COPD
Processing health outcome DIABETES
Processing health outcome ISCHEMIC_HEART_DISEASE
Processing health outcome LOWER_RESPIRATORY_INFECTIONS
Processing health outcome LUNG_CANCER
Processing health outcome STROKE
Processing health outcome DEMENTIA
All processing complete.
